This notebook demonstrates how to use the `pydantic-ai` library to create an AI agent that can solve mathematical expressions. The agent is configured with a system prompt that instructs it to act as a math professor, explain its steps, and provide a final solution.

The notebook showcases the following key features:

- **Package Installation**: Installs the necessary libraries, including `pydantic-ai` and `pydantic-graph`.
- **Imports**: Imports required modules from libraries such as `openai`, `google.colab`, `pydantic`, and `pydantic-ai`.
- **Tool Definition**: Defines three tools (`sum_numbers`, `multiply_numbers`, and `power_numbers`) using Python functions and their corresponding Pydantic models for input validation and description.
- **Agent Initialization**: Creates an `Agent` instance, providing the language model, result type, system prompt, and the defined tools.
- **Agent Execution**: Runs the agent with sample mathematical expressions to demonstrate its ability to solve them and generate step-by-step explanations.
- **Message History**: Prints the complete message history of the agent's interaction, including the user prompt, model responses, and tool calls/returns.
- **Configuration Files**: Demonstrates how to use external YAML files (`config/tools.yaml`) to define tool metadata and system prompts, promoting better organization and reusability.
- **Docstring Generation**: Introduces a `DocstringGenerator` class to dynamically generate docstrings for the tools based on the information in the YAML configuration file.
- **Function Cloning and Updating**: Shows how to clone existing functions and update their docstrings using the `DocstringGenerator`.

Overall, the notebook provides a practical example of building a tool-using AI agent with `pydantic-ai` for solving structured tasks like mathematical calculations while maintaining a conversational and explanatory style.

In [ ]:
#@title Packages Installation
!pip install -qq pydantic-ai pydantic-graph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.5/128.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.4/567.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
#@title Imports
from openai import AsyncAzureOpenAI
from google.colab import userdata

from pprint import pprint
import json
import httpx

import nest_asyncio

from dataclasses import dataclass
from IPython.display import Markdown, display, clear_output, Image
import numpy as np

from typing import Literal, Union, Tuple, List, Dict, Optional, Any

from rich.prompt import Prompt

from pydantic import BaseModel, Field
from pydantic_ai import Agent, Tool, RunContext
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.messages import ModelMessage
from pydantic_ai.result import Usage, UsageLimits
from pydantic_graph import BaseNode, End, Graph, GraphRunContext, HistoryStep

async_azure_client = AsyncAzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))
nest_asyncio.apply()
model = OpenAIModel('gpt-4o', openai_client=async_azure_client)
from rich import print
from functools import partial

In [ ]:
import numpy as np
from typing import List, Tuple
from pydantic import BaseModel, Field

class Solution(BaseModel):
    final_message: str = Field(..., description="Final message including the result")
    result: float

agent = Agent(
    model,
    result_type=str | Solution,
    system_prompt=(
        "You are a math professor who explains how to solve math problems. "
        "Always generate a plan on how to figure the user case out in detail, "
        "so please generate a plan (do not solve anything, do not include any answer) "
        "with the tools using bullet points in a stand-alone message. "
        "Mention the tools and their respective parameters. "
        "Finally, generate a final message with the solution."
    ),
)

@agent.tool_plain
def sum_numbers(numbers: List[float]) -> float:
    """
    Only sum numbers.
    """
    return np.sum(numbers).tolist()

@agent.tool_plain
def multiply_numbers(numbers: List[float]) -> float:
    """
    Only multiply numbers.
    """
    return np.prod(numbers).tolist()

@agent.tool_plain
def power_numbers(numbers: List[float]) -> float:
    """
    Only power two numbers.
    """
    return np.power(numbers[0], numbers[1]).tolist()

result = agent.run_sync('Solve: 1+2+3*4+5+6*7+8ˆ2')
print(result.data)

Solution(final_message='The result of the expression 1 + 2 + 3 * 4 + 5 + 6 * 7 + 8^2 is 126.', result=126.0)

In [ ]:
pprint(result.all_messages(),width=160)

[ModelRequest(parts=[SystemPromptPart(content='You are a math professor who explains how to solve math problems. Always generate a plan on how to figure the '
                                              'user case out in detail, so please generate a plan (do not solve anything, do not include any answer) with the '
                                              'tools using bullet points in a stand-alone message. Mention the tools and their respective parameters. Finally, '
                                              'generate a final message with the solution.',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content='Solve: 1+2+3*4+5+6*7+8ˆ2',
                                    timestamp=datetime.datetime(2025, 3, 10, 21, 55, 0, 291899, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelRespon

In [ ]:
result = agent.run_sync('Solve: (3-1)ˆ(3+4)+(3*3)', result_type=str)
print(result.data)

The final solution for the expression \((3-1)^{(3+4)} + (3*3)\) is 137.

In [ ]:
pprint(result.all_messages(),width=160)

[ModelRequest(parts=[SystemPromptPart(content='You are a math professor who explains how to solve math problems. Always generate a plan on how to figure the '
                                              'user case out in detail, so please generate a plan (do not solve anything, do not include any answer) with the '
                                              'tools using bullet points in a stand-alone message. Mention the tools and their respective parameters. Finally, '
                                              'generate a final message with the solution.',
                                      dynamic_ref=None,
                                      part_kind='system-prompt'),
                     UserPromptPart(content='Solve: (3-1)ˆ(3+4)+(3*3)',
                                    timestamp=datetime.datetime(2025, 3, 10, 21, 55, 5, 101777, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelRespon

In [ ]:
result = agent.run_sync('Solve: (3-1)ˆ(3+4)+(3*3)',)
print(result.data)
print('---------------------')
result = agent.run_sync('Solve: (3-1)ˆ(3+4)+(3*3)', result_type=str)
print(result.data)

Solution(final_message='The solution to the expression (3-1)^(3+4)+(3*3) is', result=137.0)

---------------------

The solution to the expression \((3-1)^{(3+4)} + (3 \times 3)\) is \(137\).

In [ ]:
%mkdir -p config

In [ ]:
%%writefile config/tools.yaml
- system_prompt: |
    You are a math professor who explains how to solve math problems.
    Always generate a plan on how to figure the user case out in detail,
    so please generate a plan (do not solve anything, do not include any answer)
    with the tools using bullet points in a stand-alone message.
    Mention the tools and their respective parameters.
    Finally, generate a final message with the solution.
- type: function
  function:
    name: sum_numbers
    description: |
        Suma una lista de números flotantes.
    parameters:
      type: object
      properties:
        numbers:
          type: array
          items:
            type: number
          description: Lista de números flotantes a sumar.
      required:
        - numbers
      additionalProperties: false
    returns:
      type: float
      description: La suma de todos los números en la lista.
    strict: true

- type: function
  function:
    name: multiply_numbers
    description: |
        Multiplica una lista de números flotantes.
    parameters:
      type: object
      properties:
        numbers:
          type: array
          items:
            type: number
          description: |
            Lista de números flotantes a multiplicar.
            ejemplo [.., .., ..]
      required:
        - numbers
      additionalProperties: false
    returns:
      type: float
      description: El producto de todos los números en la lista.
    strict: true

- type: function
  function:
    name: power_numbers
    description: Eleva un número flotante a la potencia de otro.
    parameters:
      type: object
      properties:
        numbers:
          type: array
          items:
            type: number
          minItems: 2
          maxItems: 2
          description: Lista de dos números flotantes donde el primero es la base y el segundo es el exponente.
      required:
        - numbers
      additionalProperties: false
    returns:
      type: float
      description: El resultado de elevar el primer número al segundo.
    strict: true

Writing config/tools.yaml


In [ ]:
# metadata = yaml.safe_load(open('config/tools.yaml', "r"))
# tools = list(filter(lambda x: 'type' in x,metadata))
# instructions = next((x.get('system_prompt') for x in metadata if 'type' not in x), None)

In [ ]:
%%writefile config/help.py
import yaml
import types
from functools import lru_cache

class DocstringGenerator:
    @staticmethod
    @lru_cache(maxsize=128)
    def load_yaml_file(yaml_file_path: str) -> list:
        return yaml.safe_load(open(yaml_file_path, "r"))

    @staticmethod
    def get_system_prompt(yaml_file_path: str) -> str:
        return next((x.get('system_prompt') for x in DocstringGenerator.load_yaml_file(yaml_file_path) if 'type' not in x), None)

    @staticmethod
    def generate_docstring(function_name: str, yaml_file_path: str) -> str:
        functions_metadata = list(filter(lambda x: 'type' in x, DocstringGenerator.load_yaml_file(yaml_file_path)))
        function_data = functions_metadata[
            [entry['function']['name'] for entry in functions_metadata].index(function_name)
        ]

        description = function_data["function"]["description"]
        parameters = function_data["function"]["parameters"]["properties"]
        return_data = function_data["function"]["returns"]

        parameters_doc = "\n".join(
            [f"    {param_name} ({param_info['type']}): {param_info['description']}"
             for param_name, param_info in parameters.items()]
        )

        docstring = f'''
{description}

Args:
{parameters_doc}

Returns:
    {return_data["type"]}: {return_data["description"]}
'''
        return docstring

    @staticmethod
    def clone_and_update_function(original_function, yaml_file_path: str) -> types.FunctionType:
        cloned_function = types.FunctionType(
            original_function.__code__,
            original_function.__globals__,
            name=original_function.__name__,
            argdefs=original_function.__defaults__,
            closure=original_function.__closure__
        )

        cloned_function.__doc__ = DocstringGenerator.generate_docstring(original_function.__name__, yaml_file_path)
        cloned_function.__annotations__ = original_function.__annotations__

        return cloned_function


Writing config/help.py


In [ ]:
%%writefile agent_tools.py
import numpy as np
from typing import List
def sum_numbers(numbers: List[float]) -> float:
    return np.sum(numbers).tolist()

def multiply_numbers(numbers: List[float]) -> float:
    return np.prod(numbers).tolist()


def power_numbers(numbers: List[float]) -> float:
    return np.power(numbers[0], numbers[1]).tolist()

Writing agent_tools.py


In [ ]:
import numpy as np
from typing import List, Tuple
from pydantic import BaseModel, Field
from config.help import DocstringGenerator
from functools import partial
from agent_tools import sum_numbers, multiply_numbers, power_numbers

fun_dec = partial(DocstringGenerator.clone_and_update_function, yaml_file_path="config/tools.yaml")
system_desc = partial(DocstringGenerator.get_system_prompt, yaml_file_path="config/tools.yaml")

class Solution(BaseModel):
    final_message: str = Field(..., description="Final message including the result in the text")
    result: float

agent = Agent(
    model,
    result_type=str | Solution,
    system_prompt=system_desc(),
    tools=[
        Tool(fun_dec(sum_numbers), takes_ctx=False),
        Tool(fun_dec(multiply_numbers), takes_ctx=False),
        Tool(fun_dec(power_numbers), takes_ctx=False),
    ],
)
result = agent.run_sync('Solve: 1+2+3*4+5+6*7+8ˆ2')
print(result.data)

Solution(final_message='The result of the expression 1+2+3*4+5+6*7+8^2 is 126.', result=126.0)

In [ ]:
print(result.all_messages())

[
    ModelRequest(
        parts=[
            SystemPromptPart(
                content='You are a math professor who explains how to solve math problems.\nAlways generate a plan 
on how to figure the user case out in detail,\nso please generate a plan (do not solve anything, do not include any
answer)\nwith the tools using bullet points in a stand-alone message.\nMention the tools and their respective 
parameters.\nFinally, generate a final message with the solution.\n',
                dynamic_ref=None,
                part_kind='system-prompt'
            ),
            UserPromptPart(
                content='Solve: 1+2+3*4+5+6*7+8ˆ2',
                timestamp=datetime.datetime(2025, 3, 14, 15, 40, 28, 640134, tzinfo=datetime.timezone.utc),
                part_kind='user-prompt'
            )
        ],
        kind='request'
    ),
    ModelResponse(
        parts=[
            TextPart(
                content='To solve the given mathematical expression \\(1 + 2 + 3 \\cdot 4 + 5 + 6 \\cdot 7 + 
8^2\\), follow these steps:\n\n1. **Identify and solve the exponentiation**:\n   - Calculate \\(8^2\\).\n\n2. 
**Identify and solve the multiplication**:\n   - Calculate \\(3 \\cdot 4\\).\n   - Calculate \\(6 \\cdot 
7\\).\n\n3. **Sum the results of multiplication and exponentiation with the other numbers**:\n   - Add the results 
obtained from the above steps along with the remaining numbers (1, 2, 5).\n\nHere’s the plan with the tools needed 
for each step:\n\n- **Exponentiation Calculation**:\n  - Tool: `functions.power_numbers`\n  - Parameters: `{ 
numbers: [8, 2] }`\n\n- **Multiplication Calculations**:\n  - Tool: `functions.multiply_numbers`\n  - Parameters: 
`{ numbers: [3, 4] }`\n  - Tool: `functions.multiply_numbers`\n  - Parameters: `{ numbers: [6, 7] }`\n\n- 
**Summation of all numbers**:\n  - Tool: `functions.sum_numbers`\n  - Parameters: `{ numbers: [1, 2, (result of 
3*4), 5, (result of 6*7), (result of 8^2)] }`\n\nNow I will perform the calculations.',
                part_kind='text'
            ),
            ToolCallPart(
                tool_name='power_numbers',
                args='{"numbers": [8, 2]}',
                tool_call_id='call_4ftV8EJrlBc4l7FLpfk7gMir',
                part_kind='tool-call'
            ),
            ToolCallPart(
                tool_name='multiply_numbers',
                args='{"numbers": [3, 4]}',
                tool_call_id='call_UTCqrEAUtpUaDTjP2cW0dQFM',
                part_kind='tool-call'
            ),
            ToolCallPart(
                tool_name='multiply_numbers',
                args='{"numbers": [6, 7]}',
                tool_call_id='call_emUbn00vI48vDfQRkcdqzBLy',
                part_kind='tool-call'
            )
        ],
        model_name='gpt-4o-2024-05-13',
        timestamp=datetime.datetime(2025, 3, 14, 15, 40, 29, tzinfo=datetime.timezone.utc),
        kind='response'
    ),
    ModelRequest(
        parts=[
            ToolReturnPart(
                tool_name='power_numbers',
                content=64.0,
                tool_call_id='call_4ftV8EJrlBc4l7FLpfk7gMir',
                timestamp=datetime.datetime(2025, 3, 14, 15, 40, 34, 884919, tzinfo=datetime.timezone.utc),
                part_kind='tool-return'
            ),
            ToolReturnPart(
                tool_name='multiply_numbers',
                content=12.0,
                tool_call_id='call_UTCqrEAUtpUaDTjP2cW0dQFM',
                timestamp=datetime.datetime(2025, 3, 14, 15, 40, 34, 884973, tzinfo=datetime.timezone.utc),
                part_kind='tool-return'
            ),
            ToolReturnPart(
                tool_name='multiply_numbers',
                content=42.0,
                tool_call_id='call_emUbn00vI48vDfQRkcdqzBLy',
                timestamp=datetime.datetime(2025, 3, 14, 15, 40, 34, 885330, tzinfo=datetime.timezone.utc),
                part_kind='tool-return'
            )
        ],
        kind='request'
    ),
    ModelResponse(
        pa